# Embudo procesal: de la captura a la sentencia

Cuantos casos llegan a cada etapa y cuantos se caen en el camino. Se
calcula cuatro veces: sobre toda la base y sobre cada grupo de delito.

La unidad de analisis es el caso, o sea una persona dentro de un proceso
(IDENTIFICACION + radicado). Un radicado puede cobijar a varias personas y
una persona puede tener varios radicados.

El delito se toma de la captura (CONDUCTA) y no de la sentencia, porque
DelitoSentencia solo existe para los condenados y filtrar por ahi haria
que el 100% de cada grupo tuviera sentencia.

Las etapas van sobre el total de casos del grupo y los dos desgloses
(tipo de medida y sentido del fallo) sobre su propio subconjunto. Los
arboles son anidados, asi que los dos hijos siempre suman el padre.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError as e:
    raise ImportError(
        "Falta matplotlib. En PowerShell:\n"
        r"  uv pip install --python C:\Users\DiDi\.venv\Scripts\python.exe matplotlib"
        "\ny después reiniciá el kernel."
    ) from e


DESCARGAS = Path.home() / "Downloads"

RUTA_BASE = DESCARGAS / "base_final.parquet"
RUTA_BASE_MUESTRA = DESCARGAS / "muestra_revision.csv"

SALIDAS = DESCARGAS / "salidas_embudo"
SALIDAS.mkdir(exist_ok=True)

COL_PROCESO = "radicado"
COL_ID = "IDENTIFICACION"
COL_DELITO_CAPTURA = "CONDUCTA"

COL_LLAVE = "llave_caso"

GRUPO_TOTAL = "Total de la base"

GRUPOS = {
    "Delitos Contra la Propiedad": {"239", "240", "244", "246", "249", "250"},
    "Crimen Violento": {"103", "104A", "104B", "105", "106", "108", "111", "119" , "128" , "135", "138", "139", "139A", "141", "141A", "141B", "205", "206", "208",
            "209", "210A", "213", "213A", "214", "217", "218", "219A", "229" , "230", "230A" , "229A"},
    "Drogas y Armas": { "376" , "365" , "366"},
}

ARCHIVO = {
    GRUPO_TOTAL: "total",
    "Delitos Contra la Propiedad": "delitos contra la propiedad",
    "Crimen Violento": "crimen violento",
    "Drogas y Armas": "drogas y armas",
}

plt.rcParams.update({
    "font.size": 11,
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
if RUTA_BASE.exists():
    data = pd.read_parquet(RUTA_BASE)

    origen = RUTA_BASE.name


else:
    data = pd.read_csv(RUTA_BASE_MUESTRA, encoding="utf-8-sig")

    origen = RUTA_BASE_MUESTRA.name


print(f"Base cargada: {origen} ({len(data):,} filas en formato largo)")

cedula_limpia = data[COL_ID].astype("string").fillna("SIN_CEDULA")

radicado_limpio = (
    data[COL_PROCESO]
    .astype("string")
    .replace(["", "nan", "<NA>", "NaN"], pd.NA)
    .fillna("SIN_RADICADO")
)

data[COL_PROCESO] = radicado_limpio

fecha_limpia = data["FECHA"].astype("string").fillna("SIN_FECHA")

data[COL_LLAVE] = cedula_limpia + " | " + radicado_limpio + " | " + fecha_limpia

casos = data.drop_duplicates(subset=[COL_LLAVE]).copy()

casos["articulo_captura"] = casos[COL_DELITO_CAPTURA].str.extract(
    r"ART[IÍ]CULO\s+(\d+[A-Z]?)\.", flags=re.IGNORECASE
)[0]

casos["grupo_captura"] = np.select(
    [casos["articulo_captura"].isin(arts) for arts in GRUPOS.values()],
    list(GRUPOS),
    default="Otros",
)

N_TOTAL = len(casos)

personas_por_proceso = casos.groupby(COL_PROCESO)[COL_ID].nunique()
procesos_por_persona = casos.groupby(COL_ID)[COL_PROCESO].nunique()
capturas_por_persona = casos.groupby(COL_ID).size()

print(f"\nCasos (persona x proceso x fecha) : {N_TOTAL:,}   <- unidad de análisis")
print(f"Procesos distintos        : {casos[COL_PROCESO].nunique():,}")
print(f"Personas distintas        : {casos[COL_ID].nunique():,}")

print(f"\nProcesos con más de una persona: "
      f"{int((personas_por_proceso > 1).sum()):,} "
      f"({(personas_por_proceso > 1).mean():.1%})")
print(f"Personas con más de un proceso : "
      f"{int((procesos_por_persona > 1).sum()):,} "
      f"({(procesos_por_persona > 1).mean():.1%})")
print(f"Personas con más de una captura: "
      f"{int((capturas_por_persona > 1).sum()):,} "
      f"({(capturas_por_persona > 1).mean():.1%})")


print("\nCasos por grupo de delito de la captura:")
print(casos["grupo_captura"].value_counts().to_string())
print(f"\nSin artículo legible en CONDUCTA: "
      f"{int(casos['articulo_captura'].isna().sum()):,}")

In [ ]:
casos.tail(10)

## 1. Funciones

Se calculan una vez y se aplican cuatro veces, al total y a cada grupo.

In [ ]:
AZUL = "#4472C4"
TEXTO = "#1a1a1a"

pct = lambda a, b: (a / b * 100) if b else 0.0


def mascaras(sub: pd.DataFrame) -> dict:
    return {
        "legalizacion": sub["FechaLegalizacion"].notna(),
        "imputacion": sub["FechaImputacion"].notna(),
        "medida": sub["TipoMedida"].gt(0),
        "intramural": sub["TipoMedida"].eq(1),
        "domiciliaria": sub["TipoMedida"].eq(2),
        "acusacion": sub["FechaAcusacion"].notna(),
        "sentencia": sub["TieneSentencia"].eq(1),
        "condenatoria": sub["TipoSentencia"].eq(1),
        "absolutoria": sub["TipoSentencia"].eq(2),
    }


def tabla_embudo(sub: pd.DataFrame, grupo: str) -> pd.DataFrame:
    m = mascaras(sub)
    n = len(sub)
    n_med, n_sen = int(m["medida"].sum()), int(m["sentencia"].sum())

    etapas = [
        ("Capturas",                    None,             n,     "total"),
        ("Con legalización de captura", m["legalizacion"], n,    "total"),
        ("Sin legalización registrada", ~m["legalizacion"], n,   "total"),
        ("Con imputación",              m["imputacion"],  n,     "total"),
        ("Sin imputación",              ~m["imputacion"], n,     "total"),
        ("Con medida de aseguramiento", m["medida"],      n,     "total"),
        ("Sin medida de aseguramiento", ~m["medida"],     n,     "total"),
        ("  Intramural",                m["intramural"],  n_med, "con medida"),
        ("  Domiciliaria",              m["domiciliaria"], n_med, "con medida"),
        ("Con escrito de acusación",    m["acusacion"],   n,     "total"),
        ("Sin escrito de acusación",    ~m["acusacion"],  n,     "total"),
        ("Con sentencia",               m["sentencia"],   n,     "total"),
        ("Sin sentencia",               ~m["sentencia"],  n,     "total"),
        ("  Condenatoria",              m["condenatoria"], n_sen, "con sentencia"),
        ("  Absolutoria",               m["absolutoria"], n_sen, "con sentencia"),
    ]

    filas = []
    for etapa, condicion, denominador, base in etapas:
        k = n if condicion is None else int(condicion.sum())
        personas = (
            sub[COL_ID].nunique() if condicion is None
            else sub.loc[condicion, COL_ID].nunique()
        )

        filas.append({
            "Grupo": grupo,
            "Etapa": etapa,
            "Casos": k,
            "%": round(k / denominador * 100, 1) if denominador else np.nan,
            "Sobre": base,
            "Personas distintas": personas,
        })

    return pd.DataFrame(filas)


def dibujar_nodo(ax, x, y, titulo, n, porcentaje=None, ancho=1.5):
    ax.plot([x - ancho / 2, x + ancho / 2], [y, y], color=AZUL, lw=1.2)
    ax.text(x, y + 0.06, titulo, ha="center", va="bottom", color=TEXTO, fontsize=11)
    ax.text(
        x, y - 0.08,
        (f"{porcentaje:.2f}%\n({n:,})" if porcentaje is not None else f"{n:,}"),
        ha="center", va="top", color=TEXTO, fontsize=10.5,
    )


def dibujar_llave(ax, x_padre, y_padre, x_hijos, y_arriba, y_abajo, ancho=1.5):
    x_medio = (x_padre + ancho / 2 + x_hijos - ancho / 2) / 2

    ax.plot([x_padre + ancho / 2, x_medio], [y_padre, y_padre], color=AZUL, lw=1.2)
    ax.plot([x_medio, x_medio], [y_abajo, y_arriba], color=AZUL, lw=1.2)

    for y in (y_arriba, y_abajo):
        ax.plot([x_medio, x_hijos - ancho / 2], [y, y], color=AZUL, lw=1.2)


def dibujar_arbol(cadena, hoja, n_raiz, titulo, archivo, ancho=1.5, figsize=(16, 9)):
    fig, ax = plt.subplots(figsize=figsize)

    paso_x, subida = 2.6, 0.80
    x, y = 0.0, 0.0

    dibujar_nodo(ax, x, y, "Capturas", n_raiz, ancho=ancho)

    for titulo_si, n_si, base, titulo_no, n_no in cadena:
        x_hijo = x + paso_x
        y_si, y_no = y + subida, y - subida

        dibujar_llave(ax, x, y, x_hijo, y_si, y_no, ancho=ancho)
        dibujar_nodo(ax, x_hijo, y_si, titulo_si, n_si, pct(n_si, base), ancho=ancho)
        dibujar_nodo(ax, x_hijo, y_no, titulo_no, n_no, pct(n_no, base), ancho=ancho)

        x, y = x_hijo, y_si
        subida *= 0.74

    titulo_a, n_a, titulo_b, n_b, base_hoja = hoja
    x_hijo = x + paso_x

    dibujar_llave(ax, x, y, x_hijo, y + subida, y - subida, ancho=ancho)
    dibujar_nodo(ax, x_hijo, y + subida, titulo_a, n_a, pct(n_a, base_hoja), ancho=ancho)
    dibujar_nodo(ax, x_hijo, y - subida, titulo_b, n_b, pct(n_b, base_hoja), ancho=ancho)

    ax.set_title(titulo, fontsize=15, weight="bold", loc="left", pad=24)
    ax.text(0, -0.04,
            f"Unidad de análisis: Persona en un proceso, con fecha de captura distinta. N = {n_raiz:,}",
            transform=ax.transAxes, fontsize=10, color="#555555")
    ax.set_xlim(-1.4, x_hijo + 1.8)
    ax.axis("off")

    plt.tight_layout()
    plt.savefig(SALIDAS / archivo, bbox_inches="tight", facecolor="white")
    plt.show()


def arboles_del_grupo(sub: pd.DataFrame, grupo: str):
    m = mascaras(sub)
    n = len(sub)

    m_leg = m["legalizacion"]
    m_imp = m_leg & m["imputacion"]
    m_acu = m_imp & m["acusacion"]
    m_sen = m_acu & m["sentencia"]
    m_med = m_imp & m["medida"]

    sufijo = ARCHIVO[grupo]
    apellido = "" if grupo == GRUPO_TOTAL else f" — {grupo}"

    dibujar_arbol(
        cadena=[
            ("Con legalización", int(m_leg.sum()), n,
             "Sin legalización", int((~m_leg).sum())),
            ("Con imputación", int(m_imp.sum()), int(m_leg.sum()),
             "Sin imputación", int((m_leg & ~m["imputacion"]).sum())),
            ("Con acusación", int(m_acu.sum()), int(m_imp.sum()),
             "Sin acusación", int((m_imp & ~m["acusacion"]).sum())),
            ("Con sentencia", int(m_sen.sum()), int(m_acu.sum()),
             "Sin sentencia", int((m_acu & ~m["sentencia"]).sum())),
        ],
        hoja=("Condenatoria", int((m_sen & m["condenatoria"]).sum()),
              "Absolutoria", int((m_sen & m["absolutoria"]).sum()),
              int(m_sen.sum())),
        n_raiz=n,
        titulo=f"De la captura a la sentencia{apellido}",
        archivo=f"arbol_embudo_{sufijo}.png",
    )

    dibujar_arbol(
        cadena=[
            ("Con legalización", int(m_leg.sum()), n,
             "Sin legalización", int((~m_leg).sum())),
            ("Con imputación", int(m_imp.sum()), int(m_leg.sum()),
             "Sin imputación", int((m_leg & ~m["imputacion"]).sum())),
            ("Con medida de aseguramiento", int(m_med.sum()), int(m_imp.sum()),
             "Sin medida de aseguramiento", int((m_imp & ~m["medida"]).sum())),
        ],
        hoja=("Intramural", int((m_med & m["intramural"]).sum()),
              "Domiciliaria", int((m_med & m["domiciliaria"]).sum()),
              int(m_med.sum())),
        n_raiz=n,
        titulo=f"De la captura a la medida de aseguramiento{apellido}",
        archivo=f"arbol_captura_medida_{sufijo}.png",
        ancho=2.2,
        figsize=(15, 8),
    )

## 2. Total de la base

In [ ]:
tabla_total = tabla_embudo(casos, GRUPO_TOTAL)

print(f"{GRUPO_TOTAL.upper()}  ({N_TOTAL:,} casos)\n")
print(tabla_total.drop(columns=["Grupo"]).set_index("Etapa").to_string())

arboles_del_grupo(casos, GRUPO_TOTAL)

## 3. Cada grupo de delito

Los mismos conteos y los mismos arboles, filtrando por el delito de la
captura. Los archivos salen con el sufijo del grupo.

In [ ]:
tablas = [tabla_total]

for grupo in GRUPOS:
    sub = casos[casos["grupo_captura"] == grupo]

    if sub.empty:
        print(f"\n{grupo.upper()}: sin casos en la base, se omite.\n")
        continue

    tablas.append(tabla_embudo(sub, grupo))

    print(f"\n{'=' * 70}")
    print(f"{grupo.upper()}  ({len(sub):,} casos)\n")
    print(tablas[-1].drop(columns=["Grupo"]).set_index("Etapa").to_string())

    arboles_del_grupo(sub, grupo)


embudo = pd.concat(tablas, ignore_index=True)
embudo.to_csv(SALIDAS / "embudo_procesal.csv", index=False, encoding="utf-8-sig")

## 4. Comparacion entre grupos

In [ ]:
comparacion = []

for grupo in [GRUPO_TOTAL] + list(GRUPOS):
    sub = casos if grupo == GRUPO_TOTAL else casos[casos["grupo_captura"] == grupo]

    if sub.empty:
        continue

    m = mascaras(sub)
    n = len(sub)
    n_med, n_sen = int(m["medida"].sum()), int(m["sentencia"].sum())

    comparacion.append({
        "Grupo": grupo,
        "Casos": n,
        "% legalizados": round(pct(int(m["legalizacion"].sum()), n), 1),
        "% imputados": round(pct(int(m["imputacion"].sum()), n), 1),
        "% con medida": round(pct(n_med, n), 1),
        "% intramural (de los que tienen medida)": round(pct(int(m["intramural"].sum()), n_med), 1),
        "% con acusación": round(pct(int(m["acusacion"].sum()), n), 1),
        "% con sentencia": round(pct(n_sen, n), 1),
        "% condenatoria (de los sentenciados)": round(pct(int(m["condenatoria"].sum()), n_sen), 1),
    })


comparacion = pd.DataFrame(comparacion).set_index("Grupo")
comparacion.to_csv(SALIDAS / "embudo_comparacion.csv", encoding="utf-8-sig")

print(comparacion.T.to_string())

print(f"\nArchivos generados en {SALIDAS}:")
for archivo in sorted(SALIDAS.iterdir()):
    print(f"  {archivo.name}")

In [ ]:
AZUL = "#4472C4"
TEXTO = "#1a1a1a"

pct = lambda a, b: (a / b * 100) if b else 0.0


def mascaras(sub: pd.DataFrame) -> dict:
    return {
        "legalizacion": sub["FechaLegalizacion"].notna(),
        "imputacion": sub["FechaImputacion"].notna(),
        "medida": sub["TipoMedida"].gt(0),
        "intramural": sub["TipoMedida"].eq(1),
        "domiciliaria": sub["TipoMedida"].eq(2),
        "acusacion": sub["FechaAcusacion"].notna(),
        "sentencia": sub["TieneSentencia"].eq(1),
        "condenatoria": sub["TipoSentencia"].eq(1),
        "absolutoria": sub["TipoSentencia"].eq(2),
    }


def tabla_embudo(sub: pd.DataFrame, grupo: str) -> pd.DataFrame:
    m = mascaras(sub)
    n = len(sub)

    n_med = int(m["medida"].sum())
    n_sen = int(m["sentencia"].sum())

    etapas = [
        ("Capturas",                    None,                  n,     "total"),
        ("Con legalización de captura", m["legalizacion"],     n,     "total"),
        ("Sin legalización registrada", ~m["legalizacion"],    n,     "total"),
        ("Con imputación",              m["imputacion"],       n,     "total"),
        ("Sin imputación",              ~m["imputacion"],      n,     "total"),
        ("Con medida de aseguramiento", m["medida"],           n,     "total"),
        ("Sin medida de aseguramiento", ~m["medida"],          n,     "total"),
        ("  Intramural",                m["intramural"],       n_med, "con medida"),
        ("  Domiciliaria",              m["domiciliaria"],     n_med, "con medida"),
        ("Con escrito de acusación",    m["acusacion"],        n,     "total"),
        ("Sin escrito de acusación",    ~m["acusacion"],       n,     "total"),
        ("Con sentencia",               m["sentencia"],        n,     "total"),
        ("Sin sentencia",               ~m["sentencia"],       n,     "total"),
        ("  Condenatoria",              m["condenatoria"],     n_sen, "con sentencia"),
        ("  Absolutoria",               m["absolutoria"],      n_sen, "con sentencia"),
    ]

    filas = []

    for etapa, condicion, denominador, base in etapas:
        k = n if condicion is None else int(condicion.sum())

        personas = (
            sub[COL_ID].nunique()
            if condicion is None
            else sub.loc[condicion, COL_ID].nunique()
        )

        filas.append({
            "Grupo": grupo,
            "Etapa": etapa,
            "Casos": k,
            "%": round(k / denominador * 100, 1) if denominador else np.nan,
            "Sobre": base,
            "Personas distintas": personas,
        })

    return pd.DataFrame(filas)


def dibujar_nodo(ax, x, y, titulo, n, porcentaje=None, ancho=1.5):
    ax.plot(
        [x - ancho / 2, x + ancho / 2],
        [y, y],
        color=AZUL,
        lw=1.2
    )

    ax.text(
        x,
        y + 0.06,
        titulo,
        ha="center",
        va="bottom",
        color=TEXTO,
        fontsize=11
    )

    ax.text(
        x,
        y - 0.08,
        (
            f"{porcentaje:.2f}%\n({n:,})"
            if porcentaje is not None
            else f"{n:,}"
        ),
        ha="center",
        va="top",
        color=TEXTO,
        fontsize=10.5,
    )


def dibujar_llave(
    ax,
    x_padre,
    y_padre,
    x_hijos,
    y_arriba,
    y_abajo,
    ancho=1.5
):
    x_medio = (
        x_padre + ancho / 2
        + x_hijos - ancho / 2
    ) / 2

    ax.plot(
        [x_padre + ancho / 2, x_medio],
        [y_padre, y_padre],
        color=AZUL,
        lw=1.2
    )

    ax.plot(
        [x_medio, x_medio],
        [y_abajo, y_arriba],
        color=AZUL,
        lw=1.2
    )

    for y in (y_arriba, y_abajo):
        ax.plot(
            [x_medio, x_hijos - ancho / 2],
            [y, y],
            color=AZUL,
            lw=1.2
        )


def dibujar_arbolv2(
    cadena,
    hoja,
    n_raiz,
    titulo,
    archivo,
    ancho=1.5,
    figsize=(18, 10),
    scrape_ramas=None
):
    fig, ax = plt.subplots(figsize=figsize)

    paso_x = 2.6
    subida = 0.80

    x = 0.0
    y = 0.0

    dibujar_nodo(
        ax,
        x,
        y,
        "Capturas",
        n_raiz,
        ancho=ancho
    )

    if scrape_ramas is not None:
        x_hijo = x + paso_x

        posiciones_y = [
            y + 1.5,
            y + 0.5,
            y - 0.5,
            y - 1.5
        ]

        x_inicio = x + ancho / 2

        x_medio = (
            x_inicio
            + x_hijo
            - ancho / 2
        ) / 2

        ax.plot(
            [x_inicio, x_medio],
            [y, y],
            color=AZUL,
            lw=1.2
        )

        ax.plot(
            [x_medio, x_medio],
            [posiciones_y[-1], posiciones_y[0]],
            color=AZUL,
            lw=1.2
        )

        for (titulo_rama, n_rama), y_rama in zip(
            scrape_ramas,
            posiciones_y
        ):
            ax.plot(
                [x_medio, x_hijo - ancho / 2],
                [y_rama, y_rama],
                color=AZUL,
                lw=1.2
            )

            dibujar_nodo(
                ax,
                x_hijo,
                y_rama,
                titulo_rama,
                n_rama,
                pct(n_rama, n_raiz),
                ancho=ancho
            )

        x = x_hijo
        y = posiciones_y[0]

        subida *= 0.74

    for titulo_si, n_si, base, titulo_no, n_no in cadena:
        x_hijo = x + paso_x

        y_si = y + subida
        y_no = y - subida

        dibujar_llave(
            ax,
            x,
            y,
            x_hijo,
            y_si,
            y_no,
            ancho=ancho
        )

        dibujar_nodo(
            ax,
            x_hijo,
            y_si,
            titulo_si,
            n_si,
            pct(n_si, base),
            ancho=ancho
        )

        dibujar_nodo(
            ax,
            x_hijo,
            y_no,
            titulo_no,
            n_no,
            pct(n_no, base),
            ancho=ancho
        )

        x = x_hijo
        y = y_si

        subida *= 0.74

    titulo_a, n_a, titulo_b, n_b, base_hoja = hoja

    x_hijo = x + paso_x

    dibujar_llave(
        ax,
        x,
        y,
        x_hijo,
        y + subida,
        y - subida,
        ancho=ancho
    )

    dibujar_nodo(
        ax,
        x_hijo,
        y + subida,
        titulo_a,
        n_a,
        pct(n_a, base_hoja),
        ancho=ancho
    )

    dibujar_nodo(
        ax,
        x_hijo,
        y - subida,
        titulo_b,
        n_b,
        pct(n_b, base_hoja),
        ancho=ancho
    )

    ax.set_title(
        titulo,
        fontsize=15,
        weight="bold",
        loc="left",
        pad=24
    )

    ax.text(
        0,
        -0.04,
        f"Unidad de análisis: Persona en un proceso, con fecha de captura distinta. N = {n_raiz:,}",
        transform=ax.transAxes,
        fontsize=10,
        color="#555555"
    )

    ax.set_xlim(
        -1.4,
        x_hijo + 1.8
    )

    ax.set_ylim(
        -2.3,
        2.3
    )

    ax.axis("off")

    plt.tight_layout()

    plt.savefig(
        SALIDAS / archivo,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()


def arboles_del_grupo(sub: pd.DataFrame, grupo: str):
    n = len(sub)

    estado_scrape = sub["scrape_estado"].fillna("Sin dato")

    m_encontrado = estado_scrape.eq("ENCONTRADO")

    c_encontrado = int(
        (estado_scrape == "ENCONTRADO").sum()
    )

    c_no_encontrado = int(
        (estado_scrape == "NO ENCONTRADO").sum()
    )

    c_privado = int(
        (estado_scrape == "PRIVADO").sum()
    )

    c_sin_radicado = int(
        (estado_scrape == "Sin radicado").sum()
    )

    sub_scrape = sub.loc[m_encontrado].copy()

    n_scrape = len(sub_scrape)

    m = mascaras(sub_scrape)

    m_leg = m["legalizacion"]

    m_imp = (
        m_leg
        & m["imputacion"]
    )

    m_acu = (
        m_imp
        & m["acusacion"]
    )

    m_sen = (
        m_acu
        & m["sentencia"]
    )

    m_med = (
        m_imp
        & m["medida"]
    )

    sufijo = ARCHIVO[grupo]

    apellido = (
        ""
        if grupo == GRUPO_TOTAL
        else f" — {grupo}"
    )

    dibujar_arbolv2(
        cadena=[
            (
                "Con legalización",
                int(m_leg.sum()),
                n_scrape,
                "Sin legalización",
                int((~m_leg).sum())
            ),

            (
                "Con imputación",
                int(m_imp.sum()),
                int(m_leg.sum()),
                "Sin imputación",
                int(
                    (
                        m_leg
                        & ~m["imputacion"]
                    ).sum()
                )
            ),

            (
                "Con acusación",
                int(m_acu.sum()),
                int(m_imp.sum()),
                "Sin acusación",
                int(
                    (
                        m_imp
                        & ~m["acusacion"]
                    ).sum()
                )
            ),

            (
                "Con sentencia",
                int(m_sen.sum()),
                int(m_acu.sum()),
                "Sin sentencia",
                int(
                    (
                        m_acu
                        & ~m["sentencia"]
                    ).sum()
                )
            ),
        ],

        hoja=(
            "Condenatoria",
            int(
                (
                    m_sen
                    & m["condenatoria"]
                ).sum()
            ),

            "Absolutoria",
            int(
                (
                    m_sen
                    & m["absolutoria"]
                ).sum()
            ),

            int(m_sen.sum())
        ),

        n_raiz=n,

        titulo=(
            f"De la captura a la sentencia"
            f"{apellido} — Verificación de Scrape"
        ),

        archivo=f"arbol_embudo_scrape_{sufijo}.png",

        ancho=2.2,

        figsize=(19, 10),

        scrape_ramas=[
            ("ENCONTRADO", c_encontrado),
            ("NO ENCONTRADO", c_no_encontrado),
            ("PRIVADO", c_privado),
            ("SIN RADICADO", c_sin_radicado),
        ]
    )

    dibujar_arbolv2(
        cadena=[
            (
                "Con legalización",
                int(m_leg.sum()),
                n_scrape,
                "Sin legalización",
                int((~m_leg).sum())
            ),

            (
                "Con imputación",
                int(m_imp.sum()),
                int(m_leg.sum()),
                "Sin imputación",
                int(
                    (
                        m_leg
                        & ~m["imputacion"]
                    ).sum()
                )
            ),

            (
                "Con medida de aseguramiento",
                int(m_med.sum()),
                int(m_imp.sum()),
                "Sin medida de aseguramiento",
                int(
                    (
                        m_imp
                        & ~m["medida"]
                    ).sum()
                )
            ),
        ],

        hoja=(
            "Intramural",
            int(
                (
                    m_med
                    & m["intramural"]
                ).sum()
            ),

            "Domiciliaria",
            int(
                (
                    m_med
                    & m["domiciliaria"]
                ).sum()
            ),

            int(m_med.sum())
        ),

        n_raiz=n,

        titulo=(
            f"De la captura a la medida de aseguramiento"
            f"{apellido} — Verificación de Scrape"
        ),

        archivo=f"arbol_captura_medida_scrape_{sufijo}.png",

        ancho=2.2,

        figsize=(18, 9),

        scrape_ramas=[
            ("ENCONTRADO", c_encontrado),
            ("NO ENCONTRADO", c_no_encontrado),
            ("PRIVADO", c_privado),
            ("SIN RADICADO", c_sin_radicado),
        ]
    )


print("Generando embudos con 4 ramas de Scrape para el Total...")

arboles_del_grupo(
    casos,
    GRUPO_TOTAL
)

print("¡Imágenes generadas y guardadas!")

In [ ]:
def dibujar_arbolv3(
    cadena,
    hoja,
    n_raiz,
    titulo,
    archivo,
    ancho=1.5,
    figsize=(22, 10)
):
    fig, ax = plt.subplots(figsize=figsize)

    paso_x = 2.8
    subida = 0.80

    x = 0.0
    y = 0.0

    dibujar_nodo(
        ax, x, y,
        "Capturas",
        n_raiz,
        ancho=ancho
    )

    xs = [x]
    ys = [y]

    for titulo_si, n_si, base, titulo_no, n_no in cadena:
        x_hijo = x + paso_x

        y_si = y + subida
        y_no = y - subida

        dibujar_llave(
            ax,
            x,
            y,
            x_hijo,
            y_si,
            y_no,
            ancho=ancho
        )

        dibujar_nodo(
            ax,
            x_hijo,
            y_si,
            titulo_si,
            n_si,
            pct(n_si, base),
            ancho=ancho
        )

        dibujar_nodo(
            ax,
            x_hijo,
            y_no,
            titulo_no,
            n_no,
            pct(n_no, base),
            ancho=ancho
        )

        xs.extend([x_hijo, x_hijo])
        ys.extend([y_si, y_no])

        x = x_hijo
        y = y_si

        subida *= 0.72

    titulo_a, n_a, titulo_b, n_b, base_hoja = hoja

    x_hijo = x + paso_x

    y_a = y + subida
    y_b = y - subida

    dibujar_llave(
        ax,
        x,
        y,
        x_hijo,
        y_a,
        y_b,
        ancho=ancho
    )

    dibujar_nodo(
        ax,
        x_hijo,
        y_a,
        titulo_a,
        n_a,
        pct(n_a, base_hoja),
        ancho=ancho
    )

    dibujar_nodo(
        ax,
        x_hijo,
        y_b,
        titulo_b,
        n_b,
        pct(n_b, base_hoja),
        ancho=ancho
    )

    xs.extend([x_hijo, x_hijo])
    ys.extend([y_a, y_b])

    ax.set_title(
        titulo,
        fontsize=15,
        weight="bold",
        loc="left",
        pad=24
    )

    ax.text(
        0,
        -0.04,
        f"Unidad de análisis: Persona en un proceso, con fecha de captura distinta. N = {n_raiz:,}",
        transform=ax.transAxes,
        fontsize=10,
        color="#555555"
    )

    margen_x = 1.5
    margen_y = 1.0

    ax.set_xlim(
        min(xs) - margen_x,
        max(xs) + margen_x
    )

    ax.set_ylim(
        min(ys) - margen_y,
        max(ys) + margen_y
    )

    ax.axis("off")

    plt.tight_layout()

    plt.savefig(
        SALIDAS / archivo,
        bbox_inches="tight",
        facecolor="white",
        dpi=200
    )

    plt.show()


def arboles_del_grupo_v3(sub: pd.DataFrame, grupo: str):
    m = mascaras(sub)
    n = len(sub)

    m_leg = m["legalizacion"]

    m_imp = m_leg & m["imputacion"]

    m_acu = m_imp & m["acusacion"]

    m_sen = m_acu & m["sentencia"]

    m_pre = m_sen & sub["TienePreacuerdo"].eq(1)

    sufijo = ARCHIVO[grupo]

    apellido = "" if grupo == GRUPO_TOTAL else f" — {grupo}"

    dibujar_arbolv3(

        cadena=[

            (
                "Con legalización de captura",
                int(m_leg.sum()),
                n,

                "Sin legalización registrada",
                int((~m_leg).sum())
            ),

            (
                "Con imputación",
                int(m_imp.sum()),
                int(m_leg.sum()),

                "Sin imputación",
                int((m_leg & ~m["imputacion"]).sum())
            ),

            (
                "Con escrito de acusación",
                int(m_acu.sum()),
                int(m_imp.sum()),

                "Sin escrito de acusación",
                int((m_imp & ~m["acusacion"]).sum())
            ),

            (
                "Con sentencia",
                int(m_sen.sum()),
                int(m_acu.sum()),

                "Sin sentencia",
                int((m_acu & ~m["sentencia"]).sum())
            ),
        ],

        hoja=(

            "Con preacuerdo",
            int(m_pre.sum()),

            "Sin preacuerdo",
            int((m_sen & ~sub["TienePreacuerdo"].eq(1)).sum()),

            int(m_sen.sum())
        ),

        n_raiz=n,

        titulo=f"De la captura al preacuerdo{apellido}",

        archivo=f"arbol_captura_preacuerdo_{sufijo}.png",

        ancho=1.8,

        figsize=(24, 10)
    )

### Notas para leer estos numeros

1. La base solo registra lo otorgado, no lo solicitado: un caso sin medida
   puede ser uno donde el fiscal no la pidio o uno donde el juez la nego.
2. El grupo sale del delito de la captura, y la calificacion puede cambiar
   durante el proceso, asi que un caso puede aparecer en homicidios y
   terminar condenado por otro delito.
3. Las variables procesales son del expediente, no de la persona. En un
   proceso con varias personas se le atribuyen a todas: si el juez impuso
   medida a uno solo de tres capturados, la base no lo distingue.
4. Un nulo no siempre significa que la etapa no ocurrio, puede haber
   ocurrido sin quedar registrada. Estos porcentajes son un piso.
5. TipoSentencia = 0 mezcla procesos sin fallo con procesos cuyo sentido no
   quedo escrito, asi que condenatorias mas absolutorias puede dar menos
   que el total con sentencia.